# Shear slab — single strain level

Analysis of **one** shear-strain level of a `shear_slab` run (a single-level run, or one level picked out of a
sweep). Set `LEVEL` and the run identifiers in **Config**, run the **sync** cell once, then run everything. All
analysis code lives in `scripts/lib/shear.py` (which reuses the statistics, readers and figure machinery of
`scripts/lib/triaxial.py`, so the shear G is formed the way the compression notebooks form M and G); this notebook
only configures it and draws the ten figures. Method notes, file list and the physics behind each estimate are
collected in **Notes** at the very end.


## 1 · Setup and computation

In [ ]:
import sys, importlib
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

LIB = Path('lib').resolve()            # scripts/lib: shear.py (all shear analysis code) + triaxial.py (shared machinery)
if str(LIB) not in sys.path:
    sys.path.insert(0, str(LIB))
import triaxial as tri
import shear as sh
tri = importlib.reload(tri)
sh = importlib.reload(sh)              # pick up edits to lib/shear.py without a kernel restart
sh.setup_style()
print('analysis code: ', LIB / 'shear.py')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  CONFIG -- the only cell to edit when switching runs
# ══════════════════════════════════════════════════════════════════════════
LEVEL = "0.1"       # the ONE shear-strain level analysed here, as a string.  It must be one of the
                    # STRAINS_LIST entries in shear_slab.batch (files are tagged _g<LEVEL>).
cfg = sh.Config(
    DATANAME    = "isolated_slab_support_periodic_5beads_tall_rho04_new_1.0_1.0_14000002_with_plates",
    INTERACTION = "1.0_1.0",          # epsSS_epsSP
    NSTEPS      = None,           # <steps> tag of the file names = NSTEPS of the batch (one tag for every level);
                                  # None resolves it from the files after the sync, an int pins it.
    RUN_ID      = "periodic_rho04_14M_shear_1",   # local folder under flow_data_local/{shear,plots/shear}
    STRAINS     = [LEVEL],
    # ---- measurement windows (the compression notebooks' values) ----
    plateau_frac      = 0.25,   # FIXED trailing fraction of the hold for the profile plateau means
    plateau_frac_auto = 0.45,   # LONGEST candidate trailing window for the series plateau (auto drift test)
    # ---- G as an increment from the gamma = 0 reference (as M and G in the compression notebooks) ----
    G_SUBTRACT_REF = True,      # needs the _ref files of shear_slab.lmp Phase 1.5 (runs since 2026-09-23);
                                # older runs fall back to absolute, flagged
    # ---- D_c transverse relaxation fit + hold-adequacy check ----
    DC_N_MODES = 5, DC_TRIM_BINS = 1, DC_SLOW_REF = 0.03, DC_TARGET_RESID = 0.01,
)
# Every other knob keeps its default (binWidth, plate_excl, wall_margin, n_curves, Ncount_min, dt_lj,
# ci_level, roll_win, P_BARO, DC_*, Expanse host/paths): see `sh.Config` in lib/shear.py, or pass them
# here as extra keyword arguments.

In [ ]:
# Pull the files this notebook reads from Expanse in ONE login (password + TOTP prompts).
# Logs in when ANY data file is missing locally (files a previous sync found absent on the
# cluster are remembered and do not re-prompt); FORCE_SYNC=True re-checks everything.
# SYNC=False never touches the network.
SYNC, FORCE_SYNC = True, False
if SYNC:
    sh.sync_from_expanse(cfg, levels=[LEVEL], force=FORCE_SYNC)

In [ ]:
# Load + compute everything (a few seconds):
#   R  -- the gamma = 0 reference state: geometry, plate planes, reference stresses (_ref files)
#   L  -- this level: strain, bulk stresses, z-profiles, series plateau, G (network + series), N1/N2, P_th, D_c, kappa
R = sh.load_reference(cfg)
L = sh.load_level(cfg, R, LEVEL)
assert L is not None, f'level {LEVEL}: core files missing -- run the sync cell'
sh._xlim_from(R)
sh.print_summary(cfg, [L])

## 2 · Figures

In [ ]:
# 1 · Strain diagnostic -- solid plate-based γ (prescribed), dashed surface-COM γ (slip check), dotted target; shaded = plateau window
sh.fig_strain(cfg, R, [L]);

In [ ]:
# 2 · Total stress evolution: σ^t_xz and the normal components / P_bath -- reference (dashed) -> hold (cividis) -> plateau (bold)
sh.fig_total_stress(cfg, R, L);

In [ ]:
# 3 · Solvent, polymer and total σ_xz(z) evolutions (the poroelastic split: the solvent carries no shear stress at rest)
sh.fig_partial_stress(cfg, R, L);

In [ ]:
# 4 · Network shear stress σ_p,xz(z): reference (γ = 0, dashed, ~0) and FINAL plateau state, 95 % bands; interior mean / γ = G
sh.fig_network_stress(cfg, R, L);

In [ ]:
# 5 · Bulk polymer shear stress history σ_p,xz(t), linear + log; green = auto-selected plateau window
sh.fig_series(cfg, R, L);

In [ ]:
# 6 · Shear modulus G (increments from γ = 0): (a) with the absolute values they replace, hollow, and the total-stress check; (b) the two G estimates alone
sh.fig_G(cfg, R, L);

In [ ]:
# 7 · Normal stress differences N1, N2 vs step (polymer and total) -- linearity / isotropy check
sh.fig_normal(cfg, R, L);

In [ ]:
# 8 · Cooperative diffusivity D_c: transverse relaxation fit of u_x(ẑ, t)/L during the hold
sh.fig_Dc(cfg, R, L);

In [ ]:
# 9 · κ = D_c/G (hydraulic permeability / viscosity) from the network and the series G -- the same κ as D_c/M in compression
sh.fig_kappa(cfg, R, L);

In [ ]:
# 10 · Thermodynamic pressure P_th = −⅓ tr(σ^t): bulk value over the hold and profile evolution (dotted = P_bath = 1.5, the compression runs' bath)
sh.fig_thermo_pressure(cfg, R, L);


## Notes

Everything below is reference material — nothing above depends on reading it.

### How to use this notebook

1. **Config**: `DATANAME`, `INTERACTION`, `NSTEPS` name the run exactly as LAMMPS tagged its output files
   (`<DATANAME>_<INTERACTION>_<NSTEPS>`); `RUN_ID` is the local folder under `flow_data_local/{shear,plots/shear}/`.
   Every production file carries a `_g<level>` tag (even a single-level run); the `*_ref*` files of the γ = 0
   window are shared across levels. `NSTEPS = None` resolves the tag from the files (`cfg.tag_for()`).
2. **Sync**: one Expanse login pulls every file the notebook reads (`sh.sync_from_expanse`, the triaxial puller).
3. **Load**: `load_reference` builds the γ = 0 state `R`, `load_level` builds the level dict `L` (strain, stresses,
   series plateau, $G$, $N_1/N_2$, $P_{\rm th}$, $D_c$, $\kappa$), `print_summary` prints the headline numbers and the
   hold-adequacy check.
4. **Figures**: one function call each; every figure is also saved as a PNG in `PLOT_DIR`.

`lib/shear.py` is shared with `shear_analysis_sweep.ipynb` so the definitions cannot drift between them, and it
imports its statistics, readers, palette and legend placement from `lib/triaxial.py`, so the shear numbers are
formed with the same code as the compression numbers.

### Files read

Into `flow_data_local/shear/<RUN_ID>/` (cluster `output_files/…`), all written by `shear_slab.lmp` (z = gap, x = shear since 2026-09-22):

| file pattern | content |
|---|---|
| `stress_tensor_{polymer,solvent}[_ref]_<sim>[_g<lvl>].dat` | bulk-integrated partial stress tensors (xx yy zz xy xz yz; `compute stress/atom NULL` → **kinetic term included**), one row per averaging block of the hold |
| `stress_profile_z_{polymer,solvent}[_ref]_…` | the same six components binned in z over the bulk region (reduced coordinate z/L_z, Ncount per bin) |
| `stress_series[_ref]_…` | fine block-averaged bulk stresses (xx yy zz xz, polymer \| solvent) every `series_freq` steps over drive + hold — the analogue of `piston_force_avg` |
| `shear_strain_…` | step, L_ref, plate separation, **plate-based γ** (drive + hold); `shear_strain_surface_…`: surface-COM γ (slip diagnostic) |
| `disp_x_polymer_…` | polymer $u_x(z,t)$ during the hold (hold-referenced), for $D_c$ |
| `plate_pressure_…`, `box_dimensions_…`, `gel_dimensions_{bb,rg}_…`, `polymer_com_…` | plate planes, box, gel geometry (whole-run and per level) |

### Geometry, bulk region, windows

* Plates are normal to $z$ on the slab's faces, driven $\pm x$; the network is periodic in $x$ and $y$. Profiles
  are $z$-binned (`binWidth` = 2 σ) over the **bulk** region, `plate_excl` = 3 σ inside each plate plane; the
  **interior** used for every mean is trimmed to `wall_margin` = 4 σ inside the plates, the compression
  notebooks' `wall_margin`. Plate planes (dash-dot) are placed `plate_excl` outside the populated bins.
* Plateau window = the last `plateau_frac` = 25 % of the hold; evolution plots show the whole hold
  (`n_curves` snapshots, cividis, final curve bold black).
* The γ = 0 **reference** is Phase 1.5 of the deck: plates frozen, `ref_avg_steps` of NVT after the
  thermalisation, several snapshots so every reference number carries a CI — the analogue of the compression
  deck's ε = 0 window. The shear deck runs no barostat on the periodic-slab input (it arrives at P* = 1.5 from
  the aniso-NPH slab; the converter deletes the solvent the plates displace), so figure 10's bulk $P_{\rm th}$
  is the check that the shear reference state is the compression runs' state.

### Strain

The strain is **prescribed**: `fix move` sets both plate positions, so γ = (x_top − x_bot − x₀)/plate_sep is exact;
the notebook uses its plateau mean (`L['gamma']`, the `held` value — the halt lands within ~0.0004 of the target
since `stress_freq` = 1000). The surface-COM strain of the polymer layer bonded to the plates (dashed in figure 1)
is a slip diagnostic: it lags γ only by the plate-bond stretch.

### Shear modulus $G$ (figures 4–6)

For simple shear of an isotropic drained network the polymer partial stress carries the network shear stress
($\sigma_{s,xz}\to 0$ at rest, figure 3), so

$$G = \frac{\Delta\langle\sigma_{p,xz}\rangle}{\gamma},\qquad \Delta\sigma = \sigma(\gamma) - \sigma(\gamma=0).$$

Two estimators, built exactly like $M_{\rm network}$ and $M_{\rm piston}$ in the compression notebooks:

* **$G_{\rm network}$** (profile): plateau-averaged $\sigma_{p,xz}(z)$ (mean over every snapshot in the last
  `plateau_frac` of the hold) over the interior bins, minus the reference interior mean, over γ; CI = 95 % t-interval
  over the bins ⊕ the reference's own interval in quadrature.
* **$G_{\rm series}$** (time series): the bulk $\sigma_{p,xz}(t)$ of `stress_series` read over the **longest drift-free
  trailing window** of the hold with the circular block bootstrap (`plateau_window`: candidates `plateau_frac_auto`
  × 9/9 … 2/9, a window passes when its halves agree within their CIs), minus the block-bootstrap mean of the
  reference series, over γ; CI in quadrature.
* **$G_{\rm total}$** (check): the same from the total $\sigma^t_{xz}$; its difference from $G_{\rm network}$ is the
  solvent's share of the shear stress, which must be ~0 in the equilibrated hold.

With `G_SUBTRACT_REF=True` (default) every estimate is an **increment** from the γ = 0 reading — a slope, like the
compression $M$ and $G$ — and the absolute stress/γ values are drawn hollow in figure 6(a). $\sigma_{xz}(\gamma=0)$
vanishes by symmetry, so the subtraction mostly propagates the reference noise into the CI; runs older than
2026-09-23 have no `_ref` files and are reported absolute, flagged.

**Comparing with the compression G.** The compression notebooks form $G = (\sigma'_{zz}-\sigma'_{xx})/2\varepsilon$ from
the lateral network-stress anisotropy of the *same* 14000002 slab at the same P* = 1.5; the shear $G$ here is the direct
measurement. Both are increments from a reference window, both average the last 25 % of a hold over wall-trimmed
interior bins with t-interval CIs, and both quote a second, independent estimator (piston / series). Compare the
network estimates first; the strain ranges differ (compression ε = 0.05–0.2 vs shear γ = 0.1–0.3), so use the sweep
notebook's stress–strain slope if either shows nonlinearity.

### Normal stress differences (figure 7)

$N_1=\sigma_{xx}-\sigma_{yy}$, $N_2=\sigma_{yy}-\sigma_{zz}$ from the bulk tensors, plateau means with t-intervals over the
snapshots; both vanish for a linear isotropic network, so $N_i/\sigma_{xz}$ (printed by the summary) measures
the departure from linearity at this γ.

### Cooperative diffusivity $D_c$ and $\kappa$ (figures 8–9)

`shear_slab.lmp` resets the displacement reference at each hold onset, so `disp_x_polymer` stores
$u_x(t)-u_x(t_{\rm hold})$. With the plates frozen it vanishes at both plates and, by the antisymmetry of the shear,
at the gap centre, which selects the even sine modes in $\hat z=(z-z_{\rm bot})/L_{\rm plates}$:

$$\frac{u_x}{L}=\sum_{k=1}^{N} A_k\left[1-e^{-4\pi^2k^2\tau}\right]\sin(2\pi k\hat z),\qquad \tau=\frac{D_c\,t}{L^2},$$

a least-squares fit in $D_c$ (`DC_BOUNDS`) with `DC_N_MODES` free amplitudes over the populated bulk bins
(≥ `Ncount_min` polymer atoms, trimmed by `DC_TRIM_BINS`). The transverse mode is set by the shear modulus,
$D_c^{\rm shear}=G/\zeta$, versus $D_c^{\rm comp}=M/\zeta$ in compression with the same friction ζ = η/k, so
**$\kappa = D_c/G$ here is the same hydraulic permeability as $\kappa = D_c/M$ there** (LJ units σ⁵/(ε τ)), and
$D_c^{\rm shear}/D_c^{\rm comp} = G/M$.

**Hold-adequacy check** (printed by `print_summary`): the slowest antisymmetric mode decays with
$\tau_1 = L^2/(4\pi^2 D_c)$; the mean residual over the plateau window for the fitted $D_c$ and for `DC_SLOW_REF`
tells whether `NSTEPS` in the batch was long enough (the shear deck holds a flat `NSTEPS` per level).

### Colours and files

Wong (2011) palette; cividis for the time gradient; the bulk region is shaded grey. Figures are saved to
`flow_data_local/plots/shear/<RUN_ID>/` with the stems `strain_diagnostic` (tagged `_g<level>`),
`total_stress_evolution`, `partial_stress_evolution`, `network_stress_final`, `shear_stress_history`,
`G_comparison`, `normal_stress_differences`, `Dc_shear_fit`, `kappa`, `thermo_pressure` (sweep notebook: `sweep_*`).

*Restructured 2026-09-23 from the 13-step `shear_analysis.ipynb` (git history) to the layout of
`triaxial_compression_single.ipynb`; the G estimators were aligned with the compression notebooks' M/G machinery,
the strain became plate-based and the γ = 0 reference window was added to the deck the same day.*
